# IQ — full production geometry validation (random weights)

This notebook runs **the actual 80-layer `IQHybridForCausalLM.forward`** at the **canonical full dimensions**: 4096 hidden, 32 rank-4 Mamba-3 MIMO layers, 12 CSA, 12 HCA, 24 Stable LatentMoE/SiTU layers (32 experts each), Block AttnRes and reasoning. The embedding and output head retain the full 201088-token geometry.

**No pretrained checkpoint or tokenizer download.** For T4 compatibility the Mamba mixer runs the rank-4 mathematical PyTorch oracle, not the production TileLang kernel; attention and MoE execute their actual IQ modules. Modules are constructed on `meta` and randomly initialized one layer at a time, preserving exact geometry without requiring 50+ GB simultaneous resident weights. FP16 on T4; BF16 only where supported.

**Runtime requirement:** In Colab choose **Runtime → Change runtime type → GPU** before running. A passing result verifies streamed full-geometry execution, **not** native CUDA MIMO kernel parity, full resident checkpoint loading, or learned language ability.


In [ ]:
import os, sys, subprocess, json, pathlib, torch
print('Python',sys.version.split()[0], 'Torch',torch.__version__)
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → GPU'
gpu = torch.device('cuda:0')
print('Accelerator:',torch.cuda.get_device_name(gpu))
print('Total GiB:',round(torch.cuda.get_device_properties(gpu).total_memory/2**30,2))
print('BF16 supported:',torch.cuda.is_bf16_supported())


In [ ]:
# Reuse the exact source branch carrying the CPU MIMO reference and full runner.
repository='https://github.com/Harqer/IQ.git'
branch='test/iq-mamba3-mimo-cpu-math-20261116'
root=pathlib.Path('/content/IQ_full_production')
if not root.exists():
    subprocess.check_call(['git','clone','--single-branch','--depth','1','--branch',branch,repository,str(root)])
else:
    subprocess.check_call(['git','-C',str(root),'fetch','origin',branch])
    subprocess.check_call(['git','-C',str(root),'checkout',branch])
    subprocess.check_call(['git','-C',str(root),'reset','--hard','FETCH_HEAD'])
# Keep Colab's CUDA-enabled torch; install only the IQ import dependencies.
subprocess.check_call([sys.executable,'-m','pip','install','-q','einops','transformers','safetensors','huggingface_hub'])
commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()
print('IQ source commit:',commit)
assert (root/'scripts/validate_full_iq_random_weights_colab.py').exists()


In [ ]:
# Executes all 80 full-width layers in a single IQ forward call, then CSA/HCA
# full-width compression probes long enough to form complete compression blocks.
# Prints progress for every layer and writes a structured pass/fail report.
env=os.environ.copy()
env['PYTHONPATH']=str(root)+os.pathsep+env.get('PYTHONPATH','')
env['IQ_FULL_GEOMETRY_REPORT']='/content/iq_full_random_report.json'
result=subprocess.run([
    sys.executable, '-u', str(root/'scripts/validate_full_iq_random_weights_colab.py'),
    '--tokens', '4'
],env=env,check=False)
report_path=pathlib.Path(env['IQ_FULL_GEOMETRY_REPORT'])
if report_path.exists():
    data=json.loads(report_path.read_text())
    print('Final status:',data.get('status'))
    print('Completed layers:',len(data.get('completed_layer_indices',[])))
    print('Component counts:',data.get('component_counts'))
    print('Peak VRAM GiB:',data.get('peak_vram_gib'))
    print('Report:',report_path)
else:
    print('No JSON report was produced (initialization/setup failure).')
assert result.returncode == 0, f'Full-geometry validation FAILED with exit code {result.returncode}; see output and report'
assert data['status']=='passed'
assert len(data['completed_layer_indices'])==80
assert len(data['moe_routed_token_counts'])==24


**Evidence:** The final cell must print `Final status: passed` and `Completed layers: 80`, with counts of Mamba=32, CSA=12, HCA=12, MoE=24. On failure, the JSON report names the last completed layer and includes the error. The report can be found at `/content/iq_full_random_report.json`. Do not treat successful setup/compilation alone as execution validation.